<a href="https://colab.research.google.com/github/ashuyad/learning-transformers/blob/main/notebooks/semantic_search_with_FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset

In [ ]:
issues_dataset = load_dataset("ashuyad1234/github-issues", split='train')

In [ ]:
issues_dataset

In [ ]:
issues_dataset[2]

In [ ]:
issues_dataset = issues_dataset.filter(
    lambda x: (x["is_pull_request"] == False and len(x["comments"]) > 0)
)
issues_dataset

In [ ]:
columns = issues_dataset.column_names
columns_to_keep = ["title", "body", "html_url", "comments"]
columns_to_remove = set(columns).difference(columns_to_keep)
issues_dataset = issues_dataset.remove_columns(columns_to_remove)
issues_dataset

In [ ]:
issues_dataset.set_format('pandas')
df = issues_dataset[:]

In [ ]:
df['comments'][1].tolist()

In [ ]:
comments_df = df.explode('comments', ignore_index=True)
comments_df.head(4)

In [ ]:
from datasets import Dataset
comments_dataset = Dataset.from_pandas(comments_df)
comments_dataset

In [ ]:
comments_dataset = comments_dataset.map(
    lambda x: { "comment_length" : len(x["comments"].split())}
)

In [ ]:
comments_dataset = comments_dataset.filter(
    lambda x: x["comment_length"] > 15
)
comments_dataset

In [ ]:
def concatenate_text(example):
  title = example["title"] if example["title"] is not None else ""
  body = example["body"] if example["body"] is not None else ""
  comments = example["comments"] if example["comments"] is not None else ""
  return {
      "text": title + "\n" + body + "\n" + comments
  }

comments_dataset = comments_dataset.map(concatenate_text)

In [ ]:
comments_dataset
comments_dataset[0]

In [ ]:
from transformers import AutoTokenizer, AutoModel

checkpoint = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model moved to: {device}")

In [ ]:
def cls_pooling(model_output):
  return model_output.last_hidden_state[:, 0]

In [ ]:
def get_embeddings(text_list):
  encoded_input = tokenizer(text_list, padding=True, truncation=True, return_tensors='pt')

  encoded_input = {k: v.to(device) for k,v in encoded_input.items()}
  model_output = model(**encoded_input)
  return cls_pooling(model_output)

In [ ]:
embedding = get_embeddings(comments_dataset['text'][0])
embedding.shape

In [ ]:
embedding_dataset = comments_dataset.map(
    lambda x: {"embeddings" : get_embeddings(x["text"]).detach().cpu().numpy()[0]}
)

In [ ]:
!pip install faiss-cpu
embedding_dataset.add_faiss_index(column='embeddings')

In [ ]:
question = "How to create dataset viewer offline ?"
question_embedding = get_embeddings([question]).cpu().detach().numpy()
question_embedding.shape

In [ ]:
score, samples = embedding_dataset.get_nearest_examples("embeddings", question_embedding, k=5)

In [ ]:
import pandas as pd

sample_df = pd.DataFrame.from_dict(samples)
sample_df["score"] = score
sample_df.sort_values("score", ascending=False, inplace=True)

In [ ]:
for _, row in sample_df.iterrows():
    print(f"COMMENT: {row.comments}")
    print(f"SCORE: {row.score}")
    print(f"TITLE: {row.title}")
    print(f"URL: {row.html_url}")
    print("=" * 50)
    print()